<a href="https://colab.research.google.com/github/brunopn-code/workflow-performance-analytics/blob/main/notebooks/03_sql_kpi_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL KPI Analysis

This notebook creates business-oriented process KPIs from the BPI Challenge 2012 event log.

The goal is to translate the exploratory analysis into reusable SQL metrics that could support a workflow performance dashboard.

The analysis focuses on:

- case duration
- delayed case rate
- activity frequency
- repeated activities
- transition frequency
- waiting time between activities
- resource workload

In [2]:
!pip install duckdb

In [3]:
import pandas as pd
import duckdb

In [4]:
df = pd.read_csv("bpi_2012_events.csv")

In [5]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="mixed",
    utc=True
)

df["case_registration_date"] = pd.to_datetime(
    df["case_registration_date"],
    format="mixed",
    utc=True
)

In [6]:
con = duckdb.connect()
con.register("events", df)

## Dataset Overview KPI

This query summarizes the size of the event log, including total events, total cases, unique activities, and unique resources.

In [7]:
con.sql("""
SELECT
    COUNT(*) AS total_events,
    COUNT(DISTINCT case_id) AS total_cases,
    COUNT(DISTINCT activity) AS total_activities,
    COUNT(DISTINCT resource) AS total_resources,
    MIN(timestamp) AS first_event_timestamp,
    MAX(timestamp) AS last_event_timestamp
FROM events
""").df()

,total_events,total_cases,total_activities,total_resources,first_event_timestamp,last_event_timestamp
0,262200,13087,24,68,2011-10-01 00:38:44.546000+00:00,2012-03-14 16:04:54.681000+00:00


## Case Duration KPI

This query calculates the duration of each loan application case from its first event to its last event.

In [13]:
case_duration_sql = con.sql("""
WITH case_times AS (
    SELECT
        case_id,
        MIN(timestamp) AS case_start,
        MAX(timestamp) AS case_end
    FROM events
    GROUP BY case_id
)

SELECT
    case_id,
    case_start,
    case_end,
    DATE_DIFF('hour', case_start, case_end) / 24.0 AS duration_days
FROM case_times
ORDER BY duration_days DESC
""").df()

case_duration_sql["duration_days"] = case_duration_sql["duration_days"].astype(int)
case_duration_sql.head()

,case_id,case_start,case_end,duration_days
0,173694,2011-10-01 08:10:30.287000+00:00,2012-02-15 12:29:26.299000+00:00,137
1,179591,2011-10-24 23:19:38.689000+00:00,2012-01-24 09:15:14.850000+00:00,91
2,188485,2011-11-23 15:56:48.528000+00:00,2012-02-19 09:15:24.248000+00:00,87
3,189805,2011-11-29 12:31:07.629000+00:00,2012-02-23 09:33:31.826000+00:00,85
4,183405,2011-11-09 11:54:58.210000+00:00,2012-02-01 18:36:24.466000+00:00,84


In [14]:
case_duration_sql["duration_days"].describe()

,duration_days
count,13087.000000
mean,8.302972
std,11.958221
min,0.000000
25%,0.000000
50%,0.000000
75%,14.000000
max,137.000000


In [15]:
con.register("case_duration", case_duration_sql)

In [16]:
case_duration_category_sql = con.sql("""
SELECT
    case_id,
    duration_days,
    CASE
        WHEN duration_days = 0 THEN 'Same day'
        WHEN duration_days <= 1 THEN '1 day'
        WHEN duration_days <= 14 THEN '2-14 days'
        WHEN duration_days <= 40 THEN '15-40 days'
        ELSE 'Over 40 days'
    END AS duration_category
FROM case_duration
""").df()

case_duration_category_sql.head()

,case_id,duration_days,duration_category
0,173694,137,Over 40 days
1,179591,91,Over 40 days
2,188485,87,Over 40 days
3,189805,85,Over 40 days
4,183405,84,Over 40 days


In [17]:
con.register("case_duration_category", case_duration_category_sql)

con.sql("""
SELECT
    duration_category,
    COUNT(*) AS total_cases,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS case_percentage
FROM case_duration_category
GROUP BY duration_category
ORDER BY total_cases DESC
""").df()

,duration_category,total_cases,case_percentage
0,Same day,6762,51.67
1,15-40 days,2884,22.04
2,2-14 days,2833,21.65
3,1 day,408,3.12
4,Over 40 days,200,1.53


## Activity Frequency KPI

This query identifies the most frequent activities in the process. Activity frequency helps show where most operational work is concentrated.

In [18]:
activity_frequency_sql = con.sql("""
SELECT
    activity,
    COUNT(*) AS total_events,
    COUNT(DISTINCT case_id) AS cases_involved,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS event_percentage
FROM events
GROUP BY activity
ORDER BY total_events DESC
""").df()

activity_frequency_sql

,activity,total_events,cases_involved,event_percentage
0,W_Completeren aanvraag,54850,7367,20.92
1,W_Nabellen offertes,52016,5015,19.84
2,W_Nabellen incomplete dossiers,25190,1647,9.61
3,W_Valideren aanvraag,20809,3254,7.94
4,W_Afhandelen leads,16566,4755,6.32
5,A_PARTLYSUBMITTED,13087,13087,4.99
6,A_SUBMITTED,13087,13087,4.99
7,A_DECLINED,7635,7635,2.91
8,A_PREACCEPTED,7367,7367,2.81
9,O_CREATED,7030,5015,2.68


## Completed Workflow Activity KPI

This query focuses on completed operational workflow tasks. These activities represent manual process work and are useful for analyzing workload and rework.

In [19]:
completed_workflow_activity_sql = con.sql("""
SELECT
    activity,
    COUNT(*) AS completed_events,
    COUNT(DISTINCT case_id) AS cases_involved
FROM events
WHERE
    activity LIKE 'W_%'
    AND LOWER(lifecycle_transition) = 'complete'
GROUP BY activity
ORDER BY completed_events DESC
""").df()

completed_workflow_activity_sql

,activity,completed_events,cases_involved
0,W_Completeren aanvraag,23967,7367
1,W_Nabellen offertes,22976,5011
2,W_Nabellen incomplete dossiers,11407,1647
3,W_Valideren aanvraag,7895,3209
4,W_Afhandelen leads,5898,4755
5,W_Beoordelen fraude,270,108


## Rework KPI

This query identifies repeated completed workflow activities within the same case. If the same completed workflow activity appears more than once in a case, it is treated as repeated work.

In [20]:
rework_sql = con.sql("""
WITH completed_work AS (
    SELECT
        case_id,
        activity,
        COUNT(*) AS activity_count
    FROM events
    WHERE
        activity LIKE 'W_%'
        AND LOWER(lifecycle_transition) = 'complete'
    GROUP BY case_id, activity
),

repeated_work AS (
    SELECT
        case_id,
        activity,
        activity_count
    FROM completed_work
    WHERE activity_count > 1
)

SELECT
    r.activity,
    COUNT(DISTINCT r.case_id) AS cases_with_rework,
    ROUND(AVG(r.activity_count), 2) AS avg_repetitions_per_reworked_case,
    MAX(r.activity_count) AS max_repetitions_in_case
FROM repeated_work r
GROUP BY r.activity
ORDER BY cases_with_rework DESC
""").df()

rework_sql

,activity,cases_with_rework,avg_repetitions_per_reworked_case,max_repetitions_in_case
0,W_Completeren aanvraag,4420,4.76,34
1,W_Nabellen offertes,4274,5.20,33
2,W_Valideren aanvraag,2020,3.32,28
3,W_Nabellen incomplete dossiers,1474,7.62,60
4,W_Afhandelen leads,772,2.48,15
5,W_Beoordelen fraude,81,3.00,9


## Rework by Duration Category KPI

This query compares repeated workflow activities across case duration categories. This helps identify which repeated activities are more common in delayed cases.

In [21]:
rework_by_duration_sql = con.sql("""
WITH completed_work AS (
    SELECT
        case_id,
        activity,
        COUNT(*) AS activity_count
    FROM events
    WHERE
        activity LIKE 'W_%'
        AND LOWER(lifecycle_transition) = 'complete'
    GROUP BY case_id, activity
),

repeated_work AS (
    SELECT
        case_id,
        activity,
        activity_count
    FROM completed_work
    WHERE activity_count > 1
),

cases_by_duration AS (
    SELECT
        duration_category,
        COUNT(DISTINCT case_id) AS total_cases
    FROM case_duration_category
    GROUP BY duration_category
)

SELECT
    c.duration_category,
    r.activity,
    COUNT(DISTINCT r.case_id) AS cases_with_rework,
    b.total_cases,
    ROUND(COUNT(DISTINCT r.case_id) * 100.0 / b.total_cases, 2) AS rework_case_rate,
    ROUND(AVG(r.activity_count), 2) AS avg_repetitions_per_reworked_case,
    MAX(r.activity_count) AS max_repetitions_in_case
FROM repeated_work r
JOIN case_duration_category c
    ON r.case_id = c.case_id
JOIN cases_by_duration b
    ON c.duration_category = b.duration_category
GROUP BY
    c.duration_category,
    r.activity,
    b.total_cases
ORDER BY
    c.duration_category,
    rework_case_rate DESC
""").df()

rework_by_duration_sql

,duration_category,activity,cases_with_rework,total_cases,rework_case_rate,avg_repetitions_per_reworked_case,max_repetitions_in_case
0,1 day,W_Completeren aanvraag,246,408,60.29,3.88,10
1,1 day,W_Afhandelen leads,45,408,11.03,2.38,6
2,1 day,W_Beoordelen fraude,18,408,4.41,2.67,7
3,1 day,W_Nabellen offertes,6,408,1.47,3.83,6
4,1 day,W_Valideren aanvraag,4,408,0.98,3.00,4
5,1 day,W_Nabellen incomplete dossiers,2,408,0.49,4.50,5
6,15-40 days,W_Nabellen offertes,2304,2884,79.89,6.14,30
7,15-40 days,W_Completeren aanvraag,1919,2884,66.54,5.84,34
8,15-40 days,W_Valideren aanvraag,988,2884,34.26,3.52,16
9,15-40 days,W_Nabellen incomplete dossiers,812,2884,28.16,8.14,45


## Waiting Time Between Activities KPI

This query calculates the waiting time between completed activities within the same case.

For each completed event, the previous completed activity and timestamp are identified using SQL window functions. The waiting time is then calculated as the time difference between the previous completed event and the current completed event.

In [22]:
waiting_times_sql = con.sql("""
WITH completed_events AS (
    SELECT
        case_id,
        activity,
        timestamp,
        LAG(activity) OVER (
            PARTITION BY case_id
            ORDER BY timestamp
        ) AS previous_activity,
        LAG(timestamp) OVER (
            PARTITION BY case_id
            ORDER BY timestamp
        ) AS previous_timestamp
    FROM events
    WHERE LOWER(lifecycle_transition) = 'complete'
),

waiting_times AS (
    SELECT
        case_id,
        previous_activity,
        activity,
        previous_timestamp,
        timestamp,
        DATE_DIFF('second', previous_timestamp, timestamp) / 3600.0 AS waiting_time_hours,
        DATE_DIFF('second', previous_timestamp, timestamp) / 86400.0 AS waiting_time_days
    FROM completed_events
    WHERE previous_activity IS NOT NULL
)

SELECT *
FROM waiting_times
ORDER BY waiting_time_hours DESC
""").df()

waiting_times_sql.head()

,case_id,previous_activity,activity,previous_timestamp,timestamp,waiting_time_hours,waiting_time_days
0,181529,W_Nabellen offertes,W_Nabellen offertes,2011-11-02 11:07:07.263000+00:00,2011-12-03 09:15:41.641000+00:00,742.142778,30.922616
1,179080,W_Completeren aanvraag,A_CANCELLED,2011-10-22 15:18:48.315000+00:00,2011-11-22 09:15:13.937000+00:00,737.940278,30.747512
2,179071,W_Completeren aanvraag,W_Completeren aanvraag,2011-10-22 15:35:37.510000+00:00,2011-11-22 09:15:11.701000+00:00,737.659444,30.735810
3,192870,W_Completeren aanvraag,W_Completeren aanvraag,2011-12-12 17:18:07.045000+00:00,2012-01-12 09:15:07.033000+00:00,735.950000,30.664583
4,186763,W_Nabellen offertes,W_Nabellen offertes,2011-12-01 18:23:57.951000+00:00,2012-01-01 09:15:32.158000+00:00,734.859722,30.619155


In [23]:
con.register("waiting_times", waiting_times_sql)

## Waiting Time by Transition KPI

This query summarizes waiting time between activity transitions. Only transitions with at least 30 occurrences are included to avoid overinterpreting rare paths.

In [24]:
waiting_time_summary_sql = con.sql("""
SELECT
    previous_activity,
    activity,
    previous_activity || ' → ' || activity AS transition,
    COUNT(*) AS transition_count,
    ROUND(AVG(waiting_time_days), 2) AS mean_wait_days,
    ROUND(MEDIAN(waiting_time_days), 2) AS median_wait_days,
    ROUND(QUANTILE_CONT(waiting_time_days, 0.95), 2) AS p95_wait_days,
    ROUND(MAX(waiting_time_days), 2) AS max_wait_days
FROM waiting_times
GROUP BY
    previous_activity,
    activity
HAVING COUNT(*) >= 30
ORDER BY median_wait_days DESC
""").df()

waiting_time_summary_sql.head(20)

,previous_activity,activity,transition,transition_count,mean_wait_days,median_wait_days,p95_wait_days,max_wait_days
0,W_Completeren aanvraag,O_SENT_BACK,W_Completeren aanvraag → O_SENT_BACK,561,5.37,5.93,8.01,11.84
1,W_Nabellen offertes,A_REGISTERED,W_Nabellen offertes → A_REGISTERED,68,2.88,3.15,6.02,6.25
2,W_Nabellen offertes,A_APPROVED,W_Nabellen offertes → A_APPROVED,165,3.08,3.12,6.07,6.96
3,W_Nabellen offertes,A_ACTIVATED,W_Nabellen offertes → A_ACTIVATED,61,2.59,2.99,6.10,6.91
4,W_Nabellen offertes,O_SENT_BACK,W_Nabellen offertes → O_SENT_BACK,2693,3.59,2.97,7.79,18.08
5,W_Nabellen offertes,O_ACCEPTED,W_Nabellen offertes → O_ACCEPTED,350,2.83,2.93,6.05,7.04
6,W_Nabellen offertes,A_DECLINED,W_Nabellen offertes → A_DECLINED,199,2.67,2.79,5.78,8.32
7,W_Nabellen offertes,O_DECLINED,W_Nabellen offertes → O_DECLINED,196,2.78,2.31,5.88,9.07
8,W_Nabellen offertes,W_Valideren aanvraag,W_Nabellen offertes → W_Valideren aanvraag,2114,2.50,2.16,5.94,7.14
9,W_Nabellen offertes,W_Nabellen offertes,W_Nabellen offertes → W_Nabellen offertes,13521,3.11,1.87,8.99,30.92


## Waiting Time by Duration Category KPI

This query compares transition waiting times across case duration categories. It helps identify which transitions become more frequent or slower in delayed cases.

In [25]:
waiting_time_by_duration_sql = con.sql("""
WITH waiting_with_duration AS (
    SELECT
        w.case_id,
        c.duration_category,
        w.previous_activity,
        w.activity,
        w.previous_activity || ' → ' || w.activity AS transition,
        w.waiting_time_days
    FROM waiting_times w
    JOIN case_duration_category c
        ON w.case_id = c.case_id
),

cases_by_duration AS (
    SELECT
        duration_category,
        COUNT(DISTINCT case_id) AS total_cases
    FROM case_duration_category
    GROUP BY duration_category
),

transition_summary AS (
    SELECT
        duration_category,
        previous_activity,
        activity,
        transition,
        COUNT(*) AS transition_count,
        ROUND(AVG(waiting_time_days), 2) AS mean_wait_days,
        ROUND(MEDIAN(waiting_time_days), 2) AS median_wait_days,
        ROUND(QUANTILE_CONT(waiting_time_days, 0.95), 2) AS p95_wait_days
    FROM waiting_with_duration
    GROUP BY
        duration_category,
        previous_activity,
        activity,
        transition
    HAVING COUNT(*) >= 30
)

SELECT
    t.duration_category,
    t.transition,
    t.transition_count,
    c.total_cases,
    ROUND(t.transition_count * 1.0 / c.total_cases, 2) AS transitions_per_case,
    t.mean_wait_days,
    t.median_wait_days,
    t.p95_wait_days
FROM transition_summary t
JOIN cases_by_duration c
    ON t.duration_category = c.duration_category
ORDER BY
    t.duration_category,
    t.median_wait_days DESC
""").df()

waiting_time_by_duration_sql.head(30)

,duration_category,transition,transition_count,total_cases,transitions_per_case,mean_wait_days,median_wait_days,p95_wait_days
0,1 day,A_PARTLYSUBMITTED → A_DECLINED,78,408,0.19,1.45,1.50,1.68
1,1 day,A_PARTLYSUBMITTED → W_Afhandelen leads,44,408,0.11,0.69,0.52,1.68
2,1 day,A_PREACCEPTED → W_Completeren aanvraag,104,408,0.25,0.55,0.50,1.65
3,1 day,W_Completeren aanvraag → A_CANCELLED,85,408,0.21,0.35,0.17,1.03
4,1 day,W_Completeren aanvraag → A_DECLINED,153,408,0.38,0.35,0.16,1.11
5,1 day,W_Afhandelen leads → W_Completeren aanvraag,142,408,0.35,0.23,0.13,0.80
6,1 day,W_Afhandelen leads → A_DECLINED,34,408,0.08,0.35,0.13,1.73
7,1 day,W_Completeren aanvraag → W_Completeren aanvraag,459,408,1.13,0.24,0.12,0.88
8,1 day,W_Afhandelen leads → A_PREACCEPTED,31,408,0.08,0.04,0.01,0.10
9,1 day,A_DECLINED → W_Afhandelen leads,92,408,0.23,0.00,0.00,0.00


In [26]:
key_transition_comparison_sql = con.sql("""
WITH waiting_with_duration AS (
    SELECT
        w.case_id,
        c.duration_category,
        w.previous_activity || ' → ' || w.activity AS transition,
        w.waiting_time_days
    FROM waiting_times w
    JOIN case_duration_category c
        ON w.case_id = c.case_id
),

cases_by_duration AS (
    SELECT
        duration_category,
        COUNT(DISTINCT case_id) AS total_cases
    FROM case_duration_category
    GROUP BY duration_category
),

transition_summary AS (
    SELECT
        duration_category,
        transition,
        COUNT(*) AS transition_count,
        ROUND(MEDIAN(waiting_time_days), 2) AS median_wait_days
    FROM waiting_with_duration
    WHERE transition IN (
        'W_Nabellen offertes → W_Nabellen offertes',
        'W_Nabellen offertes → O_SENT_BACK',
        'W_Nabellen offertes → W_Valideren aanvraag',
        'W_Completeren aanvraag → W_Nabellen offertes',
        'W_Completeren aanvraag → O_SENT_BACK',
        'W_Nabellen incomplete dossiers → O_SENT_BACK'
    )
    GROUP BY
        duration_category,
        transition
    HAVING COUNT(*) >= 30
)

SELECT
    t.duration_category,
    t.transition,
    t.transition_count,
    c.total_cases,
    ROUND(t.transition_count * 1.0 / c.total_cases, 2) AS transitions_per_case,
    t.median_wait_days
FROM transition_summary t
JOIN cases_by_duration c
    ON t.duration_category = c.duration_category
WHERE t.duration_category IN ('2-14 days', '15-40 days', 'Over 40 days')
ORDER BY
    t.transition,
    t.duration_category
""").df()

key_transition_comparison_sql

,duration_category,transition,transition_count,total_cases,transitions_per_case,median_wait_days
0,15-40 days,W_Completeren aanvraag → O_SENT_BACK,67,2884,0.02,6.11
1,2-14 days,W_Completeren aanvraag → O_SENT_BACK,461,2833,0.16,5.94
2,15-40 days,W_Completeren aanvraag → W_Nabellen offertes,1924,2884,0.67,2.06
3,2-14 days,W_Completeren aanvraag → W_Nabellen offertes,1445,2833,0.51,0.00
4,Over 40 days,W_Completeren aanvraag → W_Nabellen offertes,171,200,0.86,0.01
5,15-40 days,W_Nabellen incomplete dossiers → O_SENT_BACK,144,2884,0.05,1.16
6,2-14 days,W_Nabellen incomplete dossiers → O_SENT_BACK,32,2833,0.01,0.68
7,15-40 days,W_Nabellen offertes → O_SENT_BACK,1362,2884,0.47,2.94
8,2-14 days,W_Nabellen offertes → O_SENT_BACK,1220,2833,0.43,2.99
9,Over 40 days,W_Nabellen offertes → O_SENT_BACK,107,200,0.54,3.68


## Resource Workload KPI

This query identifies which resources handled the most completed workflow activities.

In [27]:
resource_workload_sql = con.sql("""
SELECT
    resource,
    COUNT(*) AS completed_workflow_events,
    COUNT(DISTINCT case_id) AS cases_handled,
    COUNT(DISTINCT activity) AS unique_activities_handled
FROM events
WHERE
    activity LIKE 'W_%'
    AND LOWER(lifecycle_transition) = 'complete'
    AND resource IS NOT NULL
GROUP BY resource
ORDER BY completed_workflow_events DESC
""").df()

resource_workload_sql.head(20)

,resource,completed_workflow_events,cases_handled,unique_activities_handled
0,11181.0,2992,1736,5
1,10861.0,2754,1786,5
2,11180.0,2591,1578,5
3,10913.0,2574,1664,5
4,11169.0,2507,1713,5
5,11203.0,2432,1525,5
6,11119.0,2425,1662,5
7,10909.0,2395,1555,5
8,11189.0,2238,1424,5
9,11201.0,2216,1414,5
